In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

In [12]:
dataset = fetch_openml(
    name="banknote-authentication",
    version=1,
    as_frame=True
)
df = dataset.frame
df.head()

,V1,V2,V3,V4,Class
0,3.62160,8.6661,-2.8073,-0.44699,1
1,4.54590,8.1674,-2.4586,-1.46210,1
2,3.86600,-2.6383,1.9242,0.10645,1
3,3.45660,9.5228,-4.0112,-3.59440,1
4,0.32924,-4.4552,4.5718,-0.98880,1


In [13]:
dataset.data.head()


,V1,V2,V3,V4
0,3.62160,8.6661,-2.8073,-0.44699
1,4.54590,8.1674,-2.4586,-1.46210
2,3.86600,-2.6383,1.9242,0.10645
3,3.45660,9.5228,-4.0112,-3.59440
4,0.32924,-4.4552,4.5718,-0.98880


In [10]:
dataset.target.head()

,Class
0,1
1,1
2,1
3,1
4,1


In [17]:
X = dataset.data
y = dataset.target
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   V1      1372 non-null   float64 
 1   V2      1372 non-null   float64 
 2   V3      1372 non-null   float64 
 3   V4      1372 non-null   float64 
 4   Class   1372 non-null   category
dtypes: category(1), float64(4)
memory usage: 44.5 KB


In [18]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)


In [20]:
print("Class mapping:", dict(enumerate(encoder.classes_)))
print("\nNumber of notes in each class:")
print(pd.Series(dataset.target).value_counts())

print("\nFeature descriptions:")
print(X.describe())

Class mapping: {0: '1', 1: '2'}

Number of notes in each class:
Class
1    762
2    610
Name: count, dtype: int64

Feature descriptions:
                V1           V2           V3           V4
count  1372.000000  1372.000000  1372.000000  1372.000000
mean      0.433735     1.922353     1.397627    -1.191657
std       2.842763     5.869047     4.310030     2.101013
min      -7.042100   -13.773100    -5.286100    -8.548200
25%      -1.773000    -1.708200    -1.574975    -2.413450
50%       0.496180     2.319650     0.616630    -0.586650
75%       2.821475     6.814625     3.179250     0.394810
max       6.824800    12.951600    17.927400     2.449500


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

In [24]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))




c) Evaluation at threshold 0.5
Accuracy: 0.9709090909090909
Confusion matrix:
 [[145   8]
 [  0 122]]
Precision: 0.9384615384615385
Recall: 1.0
F1: 0.9682539682539683


In [25]:
print("\nd) Log-loss:", log_loss(y_test, y_prob))



d) Log-loss: 0.05411011532995156


In [26]:
positive_class = 1
positive_col = list(encoder.classes_).index(
    encoder.classes_[positive_class]
)

prob_positive = y_prob[:, positive_col]

print("\ne) Threshold comparison")

for threshold in [0.3, 0.5, 0.7]:
    y_threshold = (prob_positive >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Accuracy:", accuracy_score(y_test, y_threshold))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_threshold))
    print("Precision:", precision_score(
        y_test, y_threshold, zero_division=0
    ))
    print("Recall:", recall_score(
        y_test, y_threshold, zero_division=0
    ))
    print("F1:", f1_score(y_test, y_threshold, zero_division=0))


e) Threshold comparison

Threshold: 0.3
Accuracy: 0.9709090909090909
Confusion matrix:
 [[145   8]
 [  0 122]]
Precision: 0.9384615384615385
Recall: 1.0
F1: 0.9682539682539683

Threshold: 0.5
Accuracy: 0.9709090909090909
Confusion matrix:
 [[145   8]
 [  0 122]]
Precision: 0.9384615384615385
Recall: 1.0
F1: 0.9682539682539683

Threshold: 0.7
Accuracy: 0.9963636363636363
Confusion matrix:
 [[152   1]
 [  0 122]]
Precision: 0.991869918699187
Recall: 1.0
F1: 0.9959183673469387
